In [1]:
import pandas as pd
import hashlib
import json
from pathlib import Path
from datetime import datetime

In [2]:
# import shutil
# from pathlib import Path
# from datetime import datetime

# # Paths
# HISTORY_PATH = Path("selection_history.json")
# TRAIN_DIR    = Path("training_sets")
# OUTPUT_DIR   = Path(".")

# # Backup folder with timestamp
# BACKUP_DIR = Path("backup_before_reset") / datetime.now().strftime("%Y%m%d_%H%M%S")
# BACKUP_DIR.mkdir(parents=True, exist_ok=True)

# # Move history file if present
# if HISTORY_PATH.exists():
#     shutil.move(str(HISTORY_PATH), BACKUP_DIR / HISTORY_PATH.name)

# # Move previous training sets
# if TRAIN_DIR.exists():
#     shutil.move(str(TRAIN_DIR), BACKUP_DIR / TRAIN_DIR.name)

# # Move any unique_sample_*.csv files
# moved_any = False
# for p in OUTPUT_DIR.glob("unique_sample_*.csv"):
#     shutil.move(str(p), BACKUP_DIR / p.name)
#     moved_any = True

# print("✅ Reset complete. Archived prior state to:", BACKUP_DIR.resolve())

import shutil
from pathlib import Path
from datetime import datetime

# Paths (fixed leading slashes)
HISTORY_PATH = Path("/home/ubuntu/TW_MultiLabel_SMP/jupyter-notebooks/selection_history.json")
TRAIN_DIR    = Path("/home/ubuntu/TW_MultiLabel_SMP/datasets/training_sets")
OUTPUT_DIR   = Path(".")

# Backup folder with timestamp
BACKUP_DIR = Path("/home/ubuntu/TW_MultiLabel_SMP/datasets/backup_before_reset") / datetime.now().strftime("%Y%m%d_%H%M%S")
BACKUP_DIR.mkdir(parents=True, exist_ok=True)

# ---- Preserve history (copy, don't move) ----
if HISTORY_PATH.exists():
    backup_history = BACKUP_DIR / HISTORY_PATH.name
    try:
        shutil.copy2(HISTORY_PATH, backup_history)
        print(f"📝 Preserved history: copied to {backup_history}")
    except Exception as e:
        print(f"⚠️ Could not copy history file: {e}")
else:
    print("ℹ️ No selection_history.json found to preserve.")

# ---- Move previous training sets to backup ----
if TRAIN_DIR.exists():
    dest = BACKUP_DIR / TRAIN_DIR.name
    try:
        shutil.move(str(TRAIN_DIR), dest)
        print(f"📦 Archived training_sets to: {dest}")
    except Exception as e:
        print(f"⚠️ Could not move training_sets: {e}")
else:
    print("ℹ️ No training_sets directory found to archive.")

# ---- Move any unique_sample_*.csv files to backup ----
moved_any = False
for p in OUTPUT_DIR.glob("unique_sample_*.csv"):
    try:
        shutil.move(str(p), BACKUP_DIR / p.name)
        print(f"📄 Archived {p.name}")
        moved_any = True
    except Exception as e:
        print(f"⚠️ Could not move {p}: {e}")

if not moved_any:
    print("ℹ️ No unique_sample_*.csv files found to archive.")

print("✅ Reset complete. Old artifacts archived. History preserved in place.")



📝 Preserved history: copied to /home/ubuntu/TW_MultiLabel_SMP/datasets/backup_before_reset/20251012_043713/selection_history.json
ℹ️ No training_sets directory found to archive.
📄 Archived unique_sample_20251010_022226.csv
✅ Reset complete. Old artifacts archived. History preserved in place.


In [3]:
def row_hash(row: pd.Series) -> str:
    """Generate a deterministic hash of a row if no explicit ID column exists."""
    obj = row.to_dict()
    normalized = {str(k): ("" if pd.isna(v) else str(v)) for k, v in obj.items()}
    payload = json.dumps(normalized, sort_keys=True, ensure_ascii=False)
    return hashlib.sha256(payload.encode("utf-8")).hexdigest()

def load_df(path: str, dataset_name: str, id_column: str = None) -> pd.DataFrame:
    df = pd.read_csv(path, low_memory=False)
    df["__dataset"] = dataset_name
    df["__source_file"] = Path(path).name

    if id_column and id_column in df.columns:
        df["unique_key"] = df[id_column].astype(str)
    else:
        df["unique_key"] = df.apply(row_hash, axis=1)

    return df


In [4]:
# Update these paths to your local copies
ABORTION_PATH = "/home/ubuntu/TW_MultiLabel_SMP/datasets/abortion_data-updated - new_abortion_related_subreddits_text_posts .csv"
MISCARRIAGE_PATH = "/home/ubuntu/TW_MultiLabel_SMP/datasets/miscarriage-data-updated - miscarriage_related_posts.csv"
HARASSMENT_PATH = "/home/ubuntu/TW_MultiLabel_SMP/datasets/sexual-harrassment-data-updated - RelevantByTitle.csv"

# History file (persists across runs)
HISTORY_PATH = Path("/home/ubuntu/TW_MultiLabel_SMP/jupyter-notebooks/selection_history.json")

# Desired counts per dataset
counts = {
    "abortion": 167,
    "miscarriage": 166,
    "harassment": 167
}

# Optional: if your CSVs have a post_id or id column
ID_COLUMN = None   # e.g. "post_id"


In [5]:
# Load CSVs
abortion_df = load_df(ABORTION_PATH, "abortion", ID_COLUMN)
miscarriage_df = load_df(MISCARRIAGE_PATH, "miscarriage", ID_COLUMN)
harassment_df = load_df(HARASSMENT_PATH, "harassment", ID_COLUMN)

# Load or initialize selection history
if HISTORY_PATH.exists():
    with open(HISTORY_PATH, "r", encoding="utf-8") as f:
        history = json.load(f)
else:
    history = {"used_keys": [], "runs": []}

used_keys = set(history.get("used_keys", []))

# Exclude previously used posts
def exclude_used(df):
    return df[~df["unique_key"].isin(used_keys)].copy()

ab_pool = exclude_used(abortion_df)
mi_pool = exclude_used(miscarriage_df)
sh_pool = exclude_used(harassment_df)

In [6]:
shortages = []
if len(ab_pool) < counts["abortion"]:
    shortages.append(f"abortion (need {counts['abortion']}, have {len(ab_pool)})")
if len(mi_pool) < counts["miscarriage"]:
    shortages.append(f"miscarriage (need {counts['miscarriage']}, have {len(mi_pool)})")
if len(sh_pool) < counts["harassment"]:
    shortages.append(f"harassment (need {counts['harassment']}, have {len(sh_pool)})")

if shortages:
    raise RuntimeError("Not enough fresh rows: " + "; ".join(shortages))

sample_ab = ab_pool.sample(n=counts["abortion"], replace=False, random_state=None)
sample_mi = mi_pool.sample(n=counts["miscarriage"], replace=False, random_state=None)
sample_sh = sh_pool.sample(n=counts["harassment"], replace=False, random_state=None)

sample_all = pd.concat([sample_ab, sample_mi, sample_sh], ignore_index=True)
sample_all = sample_all.sample(frac=1.0).reset_index(drop=True)  # shuffle


In [7]:
# Save timestamped CSV
ts = datetime.now().strftime("%Y%m%d_%H%M%S")
out_path = Path(f"unique_sample_{ts}.csv")
sample_all.to_csv(out_path, index=False)

# Update history
new_keys = sample_all["unique_key"].tolist()
history["used_keys"].extend(new_keys)
history["runs"].append({
    "timestamp": datetime.utcnow().isoformat() + "Z",
    "output_file": str(out_path),
    "counts": counts,
    "selected": len(new_keys)
})

with open(HISTORY_PATH, "w", encoding="utf-8") as f:
    json.dump(history, f, ensure_ascii=False, indent=2)

print(f"✅ Saved {len(sample_all)} posts to {out_path}")
print(f"Remaining after this run:")
print("  abortion:", len(ab_pool) - counts["abortion"])
print("  miscarriage:", len(mi_pool) - counts["miscarriage"])
print("  harassment:", len(sh_pool) - counts["harassment"])


✅ Saved 500 posts to unique_sample_20251012_043807.csv
Remaining after this run:
  abortion: 3743
  miscarriage: 320
  harassment: 4494


In [8]:
# Make a clean training label column (good for ML pipelines)
sample_all = sample_all.copy()
sample_all["label"] = sample_all["__dataset"]  # keep your original columns intact

# Create a training_sets folder
TRAIN_DIR = Path("training_sets")
TRAIN_DIR.mkdir(parents=True, exist_ok=True)

# Save a per-run training file (500 rows)
train_ts = datetime.now().strftime("%Y%m%d_%H%M%S")
train_csv = TRAIN_DIR / f"train_{train_ts}.csv"
sample_all.to_csv(train_csv, index=False)

# Also keep a stable "latest" pointer you can reference in code
latest_csv = TRAIN_DIR / "train_latest.csv"
sample_all.to_csv(latest_csv, index=False)

print(f"✅ Saved training set (500 rows): {train_csv}")
print(f"🔁 Also updated: {latest_csv}")

# (Optional) Save per-class training files for class-specific experiments
PER_CLASS_DIR = TRAIN_DIR / f"per_class_{train_ts}"
PER_CLASS_DIR.mkdir(parents=True, exist_ok=True)

for cls in sample_all["label"].unique():
    out_cls = PER_CLASS_DIR / f"{cls}_train_{train_ts}.csv"
    sample_all[sample_all["label"] == cls].to_csv(out_cls, index=False)
    print(f"• Saved {cls} subset to: {out_cls}")

# (Optional) Keep a cumulative union of everything ever sampled (good for audit/repro)
CUMULATIVE_CSV = TRAIN_DIR / "all_selected_so_far.csv"
if CUMULATIVE_CSV.exists():
    prev = pd.read_csv(CUMULATIVE_CSV, low_memory=False)
    # Use unique_key to de-dup
    combined = pd.concat([prev, sample_all], ignore_index=True)
    combined = combined.drop_duplicates(subset=["unique_key"])
else:
    combined = sample_all

combined.to_csv(CUMULATIVE_CSV, index=False)
print(f"📚 Cumulative selected-so-far updated: {CUMULATIVE_CSV}")


✅ Saved training set (500 rows): training_sets/train_20251012_043813.csv
🔁 Also updated: training_sets/train_latest.csv
• Saved harassment subset to: training_sets/per_class_20251012_043813/harassment_train_20251012_043813.csv
• Saved miscarriage subset to: training_sets/per_class_20251012_043813/miscarriage_train_20251012_043813.csv
• Saved abortion subset to: training_sets/per_class_20251012_043813/abortion_train_20251012_043813.csv
📚 Cumulative selected-so-far updated: training_sets/all_selected_so_far.csv


In [9]:
sample_all.head(10)

,id,subreddit,title,selftext,created_utc,url,Tags,__dataset,__source_file,unique_key,label
0,nljahp,assault,I've been sexually harassed and assaulted for ...,I just made a post [here](https://www.reddit.c...,2021-05-26 15:08:59,https://www.reddit.com/r/sexualassault/comment...,NaN,harassment,sexual-harrassment-data-updated - RelevantByTi...,42626de4e5de1ac2bf8534424ce26c1a4c44be53aeea2d...,harassment
1,kbayek,assault,I would rather die than see an obgyn,I already stated I’m pregnant here before &amp...,2020-12-11 20:45:47,https://www.reddit.com/r/sexualassault/comment...,NaN,harassment,sexual-harrassment-data-updated - RelevantByTi...,58477979defce3d6a25d9c7da8a0d9c14d309209bdb486...,harassment
2,1ldunz7,Miscarriage,When does it start?,Just had my first prenatal visit- the ultrasou...,2025-06-17 18:20:43,https://www.reddit.com/r/Miscarriage/comments/...,NaN,miscarriage,miscarriage-data-updated - miscarriage_related...,abb4a3a077e37afbac17d86283900125a7d82593fb1e6d...,miscarriage
3,1er6d87,abortion,No fetal heartbeat because of Local reseller p...,"Hi, I am a college student (21F) I live in the...",2024-08-13 12:16:58,https://www.reddit.com/r/abortion/comments/1er...,NaN,abortion,abortion_data-updated - new_abortion_related_s...,5bd4a855a8f0efa52104a6acd236ba5731b2a470363129...,abortion
4,1h30x8f,abortion,I regret having an abortion.,This is my first post… I originally got Reddit...,2024-11-30 1:08:49,https://www.reddit.com/r/abortion/comments/1h3...,NaN,abortion,abortion_data-updated - new_abortion_related_s...,0988b30f7c468e4bf4fec920411e59cb19fabd70739c1c...,abortion
5,k4ooyx,assault,How do I cope?,I apologise for formatting because I've only e...,2020-12-01 16:51:16,https://www.reddit.com/r/sexualassault/comment...,NaN,harassment,sexual-harrassment-data-updated - RelevantByTi...,a185db6bfb18e7e9487b2d62ab13b433f1063474d3cc76...,harassment
6,1ldos2p,Miscarriage,Has anyone had success with blood thinners + h...,"Hi everyone,\n\nI’ve had two miscarriages in t...",2025-06-17 14:36:13,https://www.reddit.com/r/Miscarriage/comments/...,NaN,miscarriage,miscarriage-data-updated - miscarriage_related...,41a31151ba7eee41bbdba7f207aa98f48707eeb807d3fe...,miscarriage
7,1llfgul,Pregnant,I’m pregnant…,I (21I she/her) have tuner‘s syndrome (monoso...,2025-06-26 23:50:37,https://www.reddit.com/r/pregnant/comments/1ll...,NaN,miscarriage,miscarriage-data-updated - miscarriage_related...,cd78afde5c71631c25e0562328d2eb0c629805246f84f6...,miscarriage
8,1ll5krl,WomensHealth,Fluid on pelvis,I am 6 months post partum and at five months p...,2025-06-26 17:06:28,https://www.reddit.com/r/WomensHealth/comments...,NaN,miscarriage,miscarriage-data-updated - miscarriage_related...,81817f9f3a7e94342d0be3f3671229bccc647dca76841a...,miscarriage
9,q5fcnx,TwoXChromosomes,Today i learnt that abortion is 14 times safer...,\nWhy isn’t this fact well known already? \n\n...,2021-10-10 19:52:14,https://www.reddit.com/r/TwoXChromosomes/comme...,NaN,abortion,abortion_data-updated - new_abortion_related_s...,064271a257d67bbe6212ef39af348f2986bbdb4f3fc1cf...,abortion
